### 1. conda 환경 생성 및 활성화
터미널에서 실행:  
conda create -n autogluon_env python=3.10 -y

conda activate autogluon_env

이후 환경 안에서 AutoGluon 설치:

pip install -U pip wheel setuptools

pip install autogluon.tabular -q 


In [1]:
import torch, lightgbm, xgboost, catboost, fastai
print("🔥 All Model Libraries Installed OK")

🔥 All Model Libraries Installed OK


In [4]:
# 파일: autogluon_run.py (예시)

import pandas as pd
from autogluon.tabular import TabularPredictor

# 1) 데이터 로드
train_path = "../data/train.tsv"
test_path = "../data/test.tsv"

# TSV 이므로 sep="\t" 사용
train_df = pd.read_csv(train_path, sep="\t")
test_df = pd.read_csv(test_path, sep="\t")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)


Train shape: (1482535, 8)
Test shape: (693359, 7)


In [5]:

train_df = train_df.drop(columns=['train_id'])
test_df = test_df.drop(columns=['test_id'])

In [6]:

# 2) 타깃 / ID 컬럼 이름 설정
TARGET_COL = "price"   # 실제 타깃 컬럼명으로 바꿔 주세요
ID_COL = None           # 제출용 ID 컬럼명으로 바꿔 주세요 (없으면 None)

# 3) AutoGluon 학습
# presets / time_limit 등은 상황에 맞게 조절
predictor = TabularPredictor(
    label=TARGET_COL,
    problem_type=None,          # 회귀/분류 자동 추론, 명시하고 싶으면 "regression"/"multiclass"/"binary"
    path="autogluon_models"     # 모델이 저장될 폴더
).fit(
    train_data=train_df,
    presets="medium_quality_faster_train",  # 빠른 실험용
    time_limit=3600,                        # 최대 1시간 (초 단위), 필요에 따라 조정
)

Preset alias specified: 'medium_quality_faster_train' maps to 'medium_quality'.
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.4.0
Python Version:     3.10.19
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.19045
CPU Count:          12
Memory Avail:       20.05 GB / 31.91 GB (62.8%)
Disk Space Avail:   350.05 GB / 465.09 GB (75.3%)
Presets specified: ['medium_quality_faster_train']
Using hyperparameters preset: hyperparameters='default'
Beginning AutoGluon training ... Time limit = 3600s
AutoGluon will save models to "c:\big20\git\big20-ML-project2-team3\MercariPriceSuggestion\src\autogluon_models"
Train Data Rows:    1482535
Train Data Columns: 6
Label Column:       price
AutoGluon infers your prediction problem is: 'regression' (because dtype of label-column == float and label-values can't be converted to int).
	Label info (max, min, mean, stddev): (2009.0, 0.0, 26.73752, 38.58607)
	If 'regressi

[1000]	valid_set's rmse: 27.61
[2000]	valid_set's rmse: 27.347
[3000]	valid_set's rmse: 27.1826
[4000]	valid_set's rmse: 27.0404
[5000]	valid_set's rmse: 27.0018
[6000]	valid_set's rmse: 26.9852
[7000]	valid_set's rmse: 26.961
[8000]	valid_set's rmse: 26.9309
[9000]	valid_set's rmse: 26.8742
[10000]	valid_set's rmse: 26.8221


	-26.8221	 = Validation score   (-root_mean_squared_error)
	273.5s	 = Training   runtime
	3.64s	 = Validation runtime
Fitting model: LightGBM ... Training model for up to 2770.92s of the 2770.92s of remaining time.
	Fitting with cpus=6, gpus=0, mem=6.8/18.2 GB


[1000]	valid_set's rmse: 27.2475
[2000]	valid_set's rmse: 27.0054
[3000]	valid_set's rmse: 26.8517
[4000]	valid_set's rmse: 26.7688
[5000]	valid_set's rmse: 26.7466
[6000]	valid_set's rmse: 26.7257
[7000]	valid_set's rmse: 26.7148
[8000]	valid_set's rmse: 26.7121
[9000]	valid_set's rmse: 26.7226
[10000]	valid_set's rmse: 26.6979


	-26.6966	 = Validation score   (-root_mean_squared_error)
	265.15s	 = Training   runtime
	3.05s	 = Validation runtime
Fitting model: RandomForestMSE ... Training model for up to 2501.63s of the 2501.63s of remaining time.
	Fitting with cpus=12, gpus=0, mem=0.9/18.2 GB
	-29.4605	 = Validation score   (-root_mean_squared_error)
	2346.23s	 = Training   runtime
	0.08s	 = Validation runtime
Fitting model: CatBoost ... Training model for up to 155.22s of the 155.22s of remaining time.
	Fitting with cpus=6, gpus=0, mem=7.5/18.1 GB
	Ran out of time, early stopping on iteration 126.
	-30.6028	 = Validation score   (-root_mean_squared_error)
	154.08s	 = Training   runtime
	0.3s	 = Validation runtime
Fitting model: WeightedEnsemble_L2 ... Training model for up to 360.00s of the 0.70s of remaining time.
	Ensemble Weights: {'LightGBM': 0.524, 'LightGBMXT': 0.333, 'RandomForestMSE': 0.143}
	-26.4053	 = Validation score   (-root_mean_squared_error)
	0.02s	 = Training   runtime
	0.0s	 = Validation ru

In [7]:

# 4) 리더보드 확인 (optional)
leaderboard_df = predictor.leaderboard(silent=True)
print(leaderboard_df.head())


                 model  score_val              eval_metric  pred_time_val  \
0  WeightedEnsemble_L2 -26.405256  root_mean_squared_error       6.772245   
1             LightGBM -26.696579  root_mean_squared_error       3.050617   
2           LightGBMXT -26.822146  root_mean_squared_error       3.639286   
3      RandomForestMSE -29.460484  root_mean_squared_error       0.081334   
4             CatBoost -30.602753  root_mean_squared_error       0.304701   

      fit_time  pred_time_val_marginal  fit_time_marginal  stack_level  \
0  2884.896380                0.001008           0.019007            2   
1   265.154015                3.050617         265.154015            1   
2   273.495707                3.639286         273.495707            1   
3  2346.227652                0.081334        2346.227652            1   
4   154.077567                0.304701         154.077567            1   

   can_infer  fit_order  
0       True          5  
1       True          2  
2       True  

In [8]:

# 5) 테스트 데이터 예측
test_preds = predictor.predict(test_df)


In [9]:

# 6) 제출 파일 생성
if ID_COL in test_df.columns:
    submission = pd.DataFrame({
        ID_COL: test_df[ID_COL],
        TARGET_COL: test_preds
    })
else:
    # ID 컬럼이 없다면 단순히 index 기반으로 생성
    submission = pd.DataFrame({
        "id": range(len(test_preds)),
        TARGET_COL: test_preds
    })

submission_path = "submission_autogluon.csv"
submission.to_csv(submission_path, index=False)
print("Saved:", submission_path)


Saved: submission_autogluon.csv
